In [10]:
import numpy as np
import nibabel as nib
import tensorflow as tf
from tensorflow.keras import layers , models
from sklearn.model_selection import train_test_split
import os
import datetime


I0000 00:00:1774883382.440386   43979 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774883382.927812   43979 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774883385.436130   43979 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
from dataset.loader import load_sample
from dataset.sampler import extract

In [7]:
import os
TEST_IMG_DIR = "/home/vatsal/projects/3D-unet-costa/costa_xyz/COSTA_dataset_v1/COSTA-Dataset-v1/imagesTs"
TEST_MASK_DIR = "/home/vatsal/projects/3D-unet-costa/costa_xyz/COSTA_dataset_v1/COSTA-Dataset-v1/labelsTs"

image_files = [f for f in sorted(os.listdir(TEST_IMG_DIR)) if f.endswith(".nii.gz")]
image_paths = [os.path.join(TEST_IMG_DIR, f) for f in image_files]
mask_paths= [os.path.join(TEST_MASK_DIR, f) for f in image_files]

In [11]:
img_path = image_paths[0]
mask_path = mask_paths[0]

# img = nib.load(img_path).get_fdata().astype(np.float32)
# gt  = nib.load(mask_path).get_fdata().astype(np.float32)

img,gt=load_sample(img_path , mask_path)

# # same preprocessing
# img = (img - np.mean(img)) / (np.std(img) + 1e-8)
# print(img.shape)
# # transpose to (D,H,W)
# img = np.transpose(img, (2,0,1))
# gt  = np.transpose(gt,  (2,0,1))

gt = (gt > 0).astype(np.float32)
print(img.shape)

(87, 233, 289)


In [12]:
ip, mp = extract_patch(img, gt, patch_size=(64,96,96))

ip = ip[np.newaxis, ..., np.newaxis]   # (1,D,H,W,1)

NameError: name 'extract_patch' is not defined

In [ ]:
def load_trained_model(model_path):
    return tf.keras.models.load_model(model_path, compile=False)

In [ ]:
model=load_trained_model("/home/vatsal/projects/3D-unet-costa/models/best_model.keras")
pred = model.predict(ip)[0, ..., 0]

In [ ]:
import matplotlib.pyplot as plt

d = ip.shape[1] // 2  # middle depth slice

inp_slice  = ip[0, d, :, :, 0]
gt_slice   = mp[d]
pred_slice = pred[d]

In [ ]:
plt.figure(figsize=(15,4))

plt.subplot(1,4,1)
plt.title("Input")
plt.imshow(inp_slice, cmap='gray')

plt.subplot(1,4,2)
plt.title("GT")
plt.imshow(gt_slice, cmap='gray')

plt.subplot(1,4,3)
plt.title("Prediction")
plt.imshow(pred_slice > 0.5, cmap='gray')

plt.subplot(1,4,4)
plt.title("Overlay")
plt.imshow(inp_slice, cmap='gray')
plt.imshow(pred_slice > 0.5, alpha=0.3)

plt.show()